# CNN Regularization with Dropout

We build the **same CNN twice** — once without dropout and once with dropout — and compare how each handles overfitting on CIFAR-10.

**Objectives:**
- See the overfitting gap (train acc vs val acc) without regularization
- Observe how Dropout reduces that gap
- Experiment with different dropout rates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load and Prepare CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Train shape: {x_train.shape}")
print(f"Test shape: {x_test.shape}")

# Show a few samples
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i])
    ax.set_title(class_names[y_train[i][0]])
    ax.axis('off')
plt.suptitle('Sample CIFAR-10 Images')
plt.show()

## 2. CNN Without Dropout

Architecture: `Conv2D -> MaxPool -> Conv2D -> MaxPool -> Flatten -> Dense -> Output`

In [ ]:
EPOCHS = 10
BATCH_SIZE = 64

model_no_dropout = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_no_dropout.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_no_dropout.summary()

In [ ]:
history_no_drop = model_no_dropout.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_test, y_test),
    verbose=1
)

## 3. CNN With Dropout

Same architecture, but with:
- `Dropout(0.25)` after each MaxPooling layer
- `Dropout(0.5)` before the output layer

In [ ]:
model_with_dropout = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_with_dropout.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_with_dropout.summary()

In [ ]:
history_with_drop = model_with_dropout.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_test, y_test),
    verbose=1
)

## 4. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Without Dropout ---
axes[0].plot(history_no_drop.history['accuracy'], label='Train Acc', linewidth=2)
axes[0].plot(history_no_drop.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[0].set_title('WITHOUT Dropout')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)
axes[0].set_ylim([0.3, 1.0])

# --- With Dropout ---
axes[1].plot(history_with_drop.history['accuracy'], label='Train Acc', linewidth=2)
axes[1].plot(history_with_drop.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[1].set_title('WITH Dropout (0.25 / 0.5)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)
axes[1].set_ylim([0.3, 1.0])

plt.suptitle('Overfitting Gap: Dropout vs No Dropout', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the overfitting gap
gap_no_drop = history_no_drop.history['accuracy'][-1] - history_no_drop.history['val_accuracy'][-1]
gap_with_drop = history_with_drop.history['accuracy'][-1] - history_with_drop.history['val_accuracy'][-1]

print(f"Overfitting gap WITHOUT dropout: {gap_no_drop:.4f}")
print(f"Overfitting gap WITH dropout:    {gap_with_drop:.4f}")
print(f"\nDropout reduced the gap by {gap_no_drop - gap_with_drop:.4f}")

## 5. Your Turn: Experiment with Dropout Rates

### TODO: Try two different dropout configurations and compare

1. **Low dropout**: Use `Dropout(0.1)` everywhere
2. **High dropout**: Use `Dropout(0.5)` everywhere

Train each for 10 epochs and report:
- Final validation accuracy
- Overfitting gap (train acc - val acc)
- Which configuration worked best?

In [ ]:
# TODO: Build and train CNN with Dropout(0.1) after every layer
# model_low_drop = keras.Sequential([...])
# model_low_drop.compile(...)
# history_low = model_low_drop.fit(...)


In [ ]:
# TODO: Build and train CNN with Dropout(0.5) after every layer
# model_high_drop = keras.Sequential([...])
# model_high_drop.compile(...)
# history_high = model_high_drop.fit(...)


In [ ]:
# TODO: Plot all 3 validation accuracy curves (0.1, 0.25/0.5, 0.5) on one chart
# TODO: Print which dropout rate gave the best val accuracy


**Your findings:**

- Best dropout config: ...
- Why: ...